# 데이터 품질 검증 (Great Expectations) — headway 마트

gold 마트(`subway_headway_by_station_tod`, `subway_service_freshness`)를 GE 1.x 로 검증한다.
Expectation 정의는 WAP audit 과 **같은 모듈**(`labs/16-data-quality/data_quality_checks.py`)을 재사용 —
배치 게이트(staging)와 탐색(gold)이 동일 규칙으로 검증된다.

In [1]:
import sys
sys.path.insert(0, "/workspace/labs/16-data-quality")
from pyspark.sql import SparkSession, functions as F

PKGS = ",".join([
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "org.apache.iceberg:iceberg-aws-bundle:1.6.1",
    "org.apache.paimon:paimon-spark-3.5_2.12:1.4.1",
    "org.apache.paimon:paimon-s3:1.4.1",
])
NETTY = "-Dorg.apache.iceberg.shaded.io.netty.noUnsafe=true -Dio.netty.noUnsafe=true"

spark = (
    SparkSession.builder.appName("data-quality")
    .config("spark.jars.packages", PKGS)
    .config("spark.driver.extraJavaOptions", NETTY)
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "rest")
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg.warehouse", "s3://warehouse/")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .getOrCreate()
)
import great_expectations as gx
from great_expectations import expectations as gxe
import pandas as pd
print("Great Expectations", gx.__version__, "| Spark", spark.version)
spark.sql("SHOW TABLES IN iceberg.gold").show(truncate=False)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.paimon#paimon-spark-3.5_2.12 added as a dependency
org.apache.paimon#paimon-s3 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2e6c8d3b-374a-4bc7-8fe4-fd2080cff1c6;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.6.1 in central
	found org.apache.paimon#paimon-spark-3.5_2.12;1.4.1 in central
	found org.apache.paimon#paimon-format;1.4.1 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found com.google.code.findbugs#jsr305;1.3.9 in central
	found org.apache.spark#spark-hive_2.12;3.5.8 in central
	found org.apache.hive#hive-common;2.3.9 in centra

Great Expectations 1.1.0 | Spark 3.5.6
+---------+-----------------------------+-----------+
|namespace|tableName                    |isTemporary|
+---------+-----------------------------+-----------+
|gold     |delay_by_station             |false      |
|gold     |delay_by_timeband            |false      |
|gold     |subway_headway_by_station_tod|false      |
|gold     |subway_service_freshness     |false      |
+---------+-----------------------------+-----------+



In [2]:
from data_quality_checks import GOLD
counts = {k: spark.table(v).count() for k, v in GOLD.items()}
pd.DataFrame({"table": list(GOLD.values()), "rows": list(counts.values())})

,table,rows
0,iceberg.gold.subway_headway_by_station_tod,1049
1,iceberg.gold.subway_service_freshness,991


## 1) 단일 테이블 검증 (headway, gold)

In [2]:
from data_quality_checks import run_ge_suite, summarize_result

ctx = gx.get_context(mode="ephemeral")
res = run_ge_suite(ctx, spark, "headway", ns="gold")
print("성공:", res.success)
pd.DataFrame(summarize_result(res))

Calculating Metrics:   0%|          | 0/74 [00:00<?, ?it/s]

성공: True


,expectation,column,success,unexpected_count
0,expect_table_row_count_to_be_between,None,True,NaN
1,expect_column_values_to_not_be_null,statn_id,True,0.0
2,expect_column_values_to_not_be_null,direction,True,0.0
3,expect_column_values_to_be_in_set,direction,True,0.0
4,expect_column_values_to_be_in_set,time_band,True,0.0
5,expect_column_values_to_be_between,cv,True,0.0
6,expect_column_values_to_be_between,p50_sec,True,0.0
7,expect_column_values_to_be_between,headway_samples,True,0.0
8,expect_column_values_to_be_between,over_1p5x_ratio,True,0.0


## 2) 전체 마트 검증 요약

In [3]:
from data_quality_checks import TABLE_KEYS

ctx2 = gx.get_context(mode="ephemeral")
summary = []
for key in TABLE_KEYS:
    res = run_ge_suite(ctx2, spark, key, ns="gold")
    s = summarize_result(res)
    summary.append({"table": key, "expectations": len(s),
                    "passed": sum(x["success"] for x in s), "success": res.success})
pd.DataFrame(summary)

26/06/25 11:30:14 WARN CacheManager: Asked to cache already cached data.


Calculating Metrics:   0%|          | 0/74 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]

,table,expectations,passed,success
0,headway,9,9,True
1,freshness,3,3,True


## 3) 오염 주입 데모 — DQ 가 잡아내는가?
정상 데이터에 규칙 위반 행을 섞으면 `success=False` 가 나와야 한다(=검증이 작동).

In [4]:
from data_quality_checks import expectations_for, GOLD

good = spark.table(GOLD["headway"])
bad_row = (good.limit(1)
           .withColumn("direction", F.lit(None).cast("string"))   # NOT NULL 위반
           .withColumn("cv", F.lit(9.9)))                          # between[0,3] 위반
bad = good.unionByName(bad_row)

ctx3 = gx.get_context(mode="ephemeral")
ds = ctx3.data_sources.add_spark(name="bad_demo")
asset = ds.add_dataframe_asset(name="bad_headway")
bd = asset.add_batch_definition_whole_dataframe(name="whole")
suite = ctx3.suites.add(gx.ExpectationSuite(name="bad_headway_suite"))
for e in expectations_for("headway"):
    suite.add_expectation(e)
res = bd.get_batch(batch_parameters={"dataframe": bad}).validate(suite)
print("overall success:", res.success, " ← False 가 정상입니다 (DQ 가 오염을 잡아냄)")
pd.DataFrame([
    {"expectation": r.expectation_config.type, "column": r.expectation_config.kwargs.get("column"),
     "success": r.success, "unexpected_count": (r.result or {}).get("unexpected_count")}
    for r in res.results
])

Calculating Metrics:   0%|          | 0/74 [00:00<?, ?it/s]

overall success: False  ← False 가 정상입니다 (DQ 가 오염을 잡아냄)


,expectation,column,success,unexpected_count
0,expect_table_row_count_to_be_between,None,True,NaN
1,expect_column_values_to_not_be_null,statn_id,True,0.0
2,expect_column_values_to_not_be_null,direction,False,1.0
3,expect_column_values_to_be_in_set,direction,True,0.0
4,expect_column_values_to_be_in_set,time_band,True,0.0
5,expect_column_values_to_be_between,cv,False,1.0
6,expect_column_values_to_be_between,p50_sec,True,0.0
7,expect_column_values_to_be_between,headway_samples,True,0.0
8,expect_column_values_to_be_between,over_1p5x_ratio,True,0.0


## 4) Spark 직접 일관성 체크 (행간 규칙)

In [5]:
from data_quality_checks import run_custom_checks
pd.DataFrame(run_custom_checks(spark, ns="gold"))

,check,checked,violations
0,p90_sec >= p50_sec,1410,0
1,mean_sec > 0,1410,0
2,grain 유일성(중복 그룹 없음),1410,0
